# KX2 proximity sensor

Complete [hello world](hello-world.ipynb) through setup first. These cells assume a connected `kx2`. The beam between the fingers is wired to input 4 on the Z drive. `True` means the beam is interrupted.

In [ ]:
await kx2.sense_proximity()

## Wait for the beam

This observes the sensor without moving any axis. It returns `False` if the deadline expires.

In [ ]:
await kx2.wait_for_proximity_sensor(state=True, timeout=5)

## Search downward

Measure the search range and clear the approach path. This example searches 5 mm below the measured Z. The search first moves to `z_start`, then descends towards `z_end`. It stops when the beam trips and returns the measured Z. The input's original logic is restored even on cancellation; unrelated drive faults propagate.

In [ ]:
from pylabrobot.paa.kx2 import Axis

position = await kx2.request_joint_state()
found_z = await kx2.find_z_with_proximity_sensor(
  z_start=position[Axis.Z], z_end=position[Axis.Z] - 5,
  max_gripper_speed=5, max_gripper_acceleration=20,
)
found_z

## Cartesian search

`find_with_proximity_sensor(start, end, max_gripper_speed=..., max_gripper_acceleration=...)` follows a streamed straight segment while retaining the measured yaw. It polls the sensor on the host, so it has different stopping latency from the drive-local input stop used by the Z search. Read the [streaming limitations](continuous-motion.ipynb) and verify clearance and stopping distance before using it.

## Disconnect

In [ ]:
await kx2.stop()